# Tahap 4 - Exploratory Data Analysis

**Tujuan:** menjawab pertanyaan yang bisa dijawab dengan angka dan grafik. Seluruh angka di bawah dihitung dari `outputs/tiktok_clean.csv`.

Aturan yang dipakai: **median** untuk ringkasan (distribusi sangat condong ke kanan), **like_rate** sebagai proxy interaksi, dan setiap grafik punya judul, label sumbu, serta satu kalimat interpretasi.

In [ ]:
# Sel 1: Siapkan lingkungan & muat data bersih
import os, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = '/content'
sns.set_theme(style='whitegrid', context='notebook')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']
sns.set_palette(PALETTE)
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.titlesize': 14,
                     'axes.titleweight': 'bold', 'axes.labelsize': 11})

df = pd.read_csv(os.path.join(BASE, 'outputs', 'tiktok_clean.csv'),
                 encoding='utf-8-sig', dtype={'sound_id': 'string'})
os.makedirs(os.path.join(BASE, 'images'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'outputs'), exist_ok=True)

def simpan(nama):
    path = os.path.join(BASE, 'images', nama)
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()

print('Data bersih:', df.shape)

In [ ]:
# Sel 2: Distribusi views - kenapa median dipakai
print('=== DISTRIBUSI VIEWS ===')
print('median :', f'{df.views.median():,.0f}')
print('mean   :', f'{df.views.mean():,.0f}')
print('p90    :', f'{df.views.quantile(.9):,.0f}')
print('maks   :', f'{df.views.max():,.0f}')

plt.figure()
plt.hist(np.log10(df['views']), bins=30, color=PALETTE[0], edgecolor='white')
plt.axvline(np.log10(df['views'].median()), color=PALETTE[3],
            linestyle='--', linewidth=2, label='Median')
plt.axvline(np.log10(df['views'].mean()), color=PALETTE[1],
            linestyle=':', linewidth=2, label='Mean')
plt.title('Distribusi Views (Skala Log10)')
plt.xlabel('log10(views)')
plt.ylabel('Jumlah video')
plt.legend()
simpan('04_distribusi_views.png')
print('Interpretasi: mean tertarik ke kanan oleh sedikit video viral, sehingga median lebih mewakili video umum.')

In [ ]:
# Sel 3: Pertanyaan 2 - durasi vs interaksi
urutan_durasi = ['<15 detik', '15-30 detik', '30-60 detik', '≥60 detik', 'tidak diketahui']
dur = df.groupby('duration_group').agg(
    n_video=('video_id', 'size'),
    median_views=('views', 'median'),
    median_like_rate=('like_rate', 'median')).reindex(urutan_durasi).reset_index()
dur['median_like_pct'] = (dur['median_like_rate'] * 100).round(2)
print(dur[['duration_group', 'n_video', 'median_views', 'median_like_pct']].to_string(index=False))
dur.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_durasi.csv'), index=False, encoding='utf-8-sig')

plt.figure()
ax = sns.barplot(data=dur[dur.duration_group != 'tidak diketahui'],
                 x='duration_group', y='median_like_pct', color=PALETTE[0])
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f%%', padding=3)
plt.title('Like Rate Median per Kelompok Durasi')
plt.xlabel('Kelompok durasi')
plt.ylabel('Like rate median (%)')
plt.ylim(0, dur['median_like_pct'].max() * 1.25)
simpan('04_durasi_vs_like_rate.png')
print('Interpretasi: durasi 15-30 detik memiliki like rate median tertinggi (7,60%) sekaligus median views tertinggi di antara kelompok berdurasi jelas.')

In [ ]:
# Sel 4: Pertanyaan 3 - hari & jam posting
urutan_hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
hari = df.groupby('post_day').agg(
    n_video=('video_id', 'size'),
    median_views=('views', 'median'),
    median_like_rate=('like_rate', 'median')).reindex(urutan_hari).reset_index()
jam = df.groupby('post_hour').agg(
    n_video=('video_id', 'size'),
    median_views=('views', 'median'),
    median_like_rate=('like_rate', 'median')).reindex(range(24)).reset_index()
hari.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_hari.csv'), index=False, encoding='utf-8-sig')
jam.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_jam.csv'), index=False, encoding='utf-8-sig')
print('=== HARI ===')
print(hari.assign(median_like_pct=(hari.median_like_rate*100).round(2))[
      ['post_day','n_video','median_views','median_like_pct']].to_string(index=False))
print()
print('=== JAM (n >= 10 saja untuk klaim) ===')
print(jam[jam.n_video >= 10].assign(median_like_pct=(jam.median_like_rate*100).round(2))[
      ['post_hour','n_video','median_views','median_like_pct']].to_string(index=False))

plt.figure()
ax = sns.barplot(data=hari, x='post_day', y=hari['median_like_rate']*100, color=PALETTE[2])
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f%%', padding=3)
plt.title('Like Rate Median per Hari Posting')
plt.xlabel('Hari posting')
plt.ylabel('Like rate median (%)')
simpan('04_hari_posting.png')
print('Interpretasi: Rabu punya like rate median terendah (3,82%) dengan sampel terbanyak (236), sedangkan Selasa tertinggi (7,02%) dengan sampel 68; perbedaan bisa dipengaruhi cara data dikumpulkan.')

plt.figure(figsize=(12, 5))
sns.barplot(data=jam, x='post_hour', y=jam['median_like_rate']*100, color=PALETTE[0])
plt.title('Like Rate Median per Jam Posting')
plt.xlabel('Jam posting (zona waktu tidak tercantum di dataset)')
plt.ylabel('Like rate median (%)')
simpan('04_jam_posting.png')
print('Interpretasi: jam 22 menyumbang 173 video dengan like rate rendah (3,59%), kemungkinan artefak pengumpulan data, bukan bukti jam tersebut buruk.')

In [ ]:
# Sel 5: Pertanyaan 1 & 4 - hashtag (data-driven)
dummies = df['hashtags_clean'].str.get_dummies(sep=', ')
baris = []
for tag in dummies.columns:
    n = int(dummies[tag].sum())
    if n >= 5:
        m = dummies[tag].astype(bool)
        baris.append({'hashtag': tag, 'n_video': n,
                      'median_views': df.loc[m, 'views'].median(),
                      'median_like_pct': round(df.loc[m, 'like_rate'].median()*100, 2)})
ht = pd.DataFrame(baris)
print('=== HASHTAG Paling sering (n >= 5) ===')
print(ht.nlargest(10, 'n_video').to_string(index=False))
print()
print('=== HASHTAG dengan like rate median tertinggi (n >= 5) ===')
print(ht.nlargest(10, 'median_like_pct').to_string(index=False))
ht.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_hashtag.csv'), index=False, encoding='utf-8-sig')

plt.figure(figsize=(10, 7))
top_ht = ht.nlargest(10, 'median_like_pct').sort_values('median_like_pct')
ax = sns.barplot(data=top_ht, x='median_like_pct', y='hashtag', color=PALETTE[2])
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f%%', padding=3)
plt.title('Top 10 Hashtag berdasarkan Like Rate Median (n >= 5)')
plt.xlabel('Like rate median (%)')
plt.ylabel('Hashtag')
simpan('04_top_hashtag.png')
print('Interpretasi: hashtag dengan like rate tinggi belum tentu paling sering dipakai; frekuensi dan kualitas interaksi berbeda.')

In [ ]:
# Sel 6: Kategori konten (HEURISTIK dari kata kunci, bukan kolom resmi dataset)
teks = (df['hashtags_clean'].fillna('') + ' ' + df['caption_masked'].fillna('')).str.lower()
KATEGORI = {
    'Komedi & humor': ['funny', 'humor', 'comedy', 'joke', 'prank', 'meme', 'lol',
                       'шутка', 'пранк', 'юмор', 'смешно', 'комедия', 'смех'],
    'Edukasi & tips': ['education', 'tutorial', 'tips', 'learn', 'guide',
                       'рекомендации', 'советы', 'обучение', 'наука'],
    'Hewan & pet': ['cat', 'dog', 'animal', 'pet', 'parrot', 'crow',
                    'кот', 'кошка', 'собака', 'попугай', 'ворон', 'зоо', 'животн'],
    'Olahraga': ['tennis', 'football', 'sport', 'basketball', 'atp', 'nba',
                 'спорт', 'футбол', 'теннис', 'хоккей'],
    'Musik & tari': ['music', 'dance', 'song', 'singer', 'remix',
                     'музыка', 'танец', 'песня', 'танцы'],
    'Makanan': ['food', 'cooking', 'recipe', 'еда', 'готовка', 'рецепт', 'кухня'],
    'Berita & isu': ['news', 'politics', 'police', 'arrested', 'crime',
                     'новости', 'политика', 'полиция'],
}

def pilih_kategori(s):
    for nama, kata in KATEGORI.items():
        if any(k in s for k in kata):
            return nama
    return 'Lainnya'

df['kategori_heuristik'] = teks.apply(pilih_kategori)
kat = df.groupby('kategori_heuristik').agg(
    n_video=('video_id', 'size'),
    median_views=('views', 'median'),
    median_like_rate=('like_rate', 'median')).sort_values('n_video', ascending=False).reset_index()
kat['median_like_pct'] = (kat['median_like_rate']*100).round(2)
print('=== KATEGORI HEURISTIK ===')
print(kat.to_string(index=False))
print('Cakupan terklasifikasi:', round((1 - (kat.set_index('kategori_heuristik').loc['Lainnya','n_video']/len(df)))*100, 1), '%')
kat.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_kategori.csv'), index=False, encoding='utf-8-sig')

plt.figure(figsize=(11, 6))
ax = sns.barplot(data=kat, x='median_like_pct', y='kategori_heuristik', color=PALETTE[3])
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f%%', padding=3)
plt.title('Like Rate Median per Kategori Konten (Heuristik Kata Kunci)')
plt.xlabel('Like rate median (%)')
plt.ylabel('Kategori (dari hashtag + caption)')
simpan('04_kategori_konten.png')
print('Interpretasi: kategori ini TESAURUS buatan sendiri dan hanya mencakup 31,7% video, jadi bukan label resmi TikTok.')

In [ ]:
# Sel 7: Pertanyaan 4 lanjutan - apa yang muncul di video interaksi tinggi?
q1, q3 = df['like_rate'].quantile([.25, .75])
hi = df[df['like_rate'] >= q3]
lo = df[df['like_rate'] <= q1]
print('Ambang: q25 =', round(q1, 4), '| q75 =', round(q3, 4), '| hi =', len(hi), '| lo =', len(lo))

# Hashtag: bandingkan kemunculan di grup atas vs bawah (dengan smoothing).
rows = []
for tag in dummies.columns:
    a = int(dummies.loc[hi.index, tag].sum())
    b = int(dummies.loc[lo.index, tag].sum())
    if a >= 5:
        lift = (a / len(hi)) / ((b + 0.5) / len(lo))
        rows.append({'item': tag, 'jenis': 'hashtag', 'n_atas': a, 'n_bawah': b, 'lift': round(lift, 2)})

# Sound ID:
atas_sound = hi['sound_id'].value_counts()
bawah_sound = lo['sound_id'].value_counts()
for sid, a in atas_sound.items():
    if a >= 5:
        b = int(bawah_sound.get(sid, 0))
        rows.append({'item': str(sid), 'jenis': 'sound_id', 'n_atas': int(a),
                     'n_bawah': b, 'lift': round((a/len(hi))/((b+0.5)/len(lo)), 2)})

# Kata kunci caption (memakai daftar kata dari KATEGORI, dicari HANYA di caption).
kata_datar = sorted({k for kata in KATEGORI.values() for k in kata})
cap_hi = hi['caption_masked'].fillna('').str.lower()
cap_lo = lo['caption_masked'].fillna('').str.lower()
for kata in kata_datar:
    a = int(cap_hi.str.contains(kata, regex=False).sum())
    b = int(cap_lo.str.contains(kata, regex=False).sum())
    if a >= 5:
        lift = (a / len(hi)) / ((b + 0.5) / len(lo))
        rows.append({'item': kata, 'jenis': 'kata_caption', 'n_atas': a,
                     'n_bawah': b, 'lift': round(lift, 2)})

lift_df = pd.DataFrame(rows).sort_values('lift', ascending=False)
print('=== ITEM PALING MENONJOL DI GRUP ATAS ===')
print(lift_df.head(10).to_string(index=False))
print('=== KATA CAPTION PALING MENONJOL DI GRUP ATAS ===')
kat_cap = lift_df[lift_df.jenis == 'kata_caption']
if kat_cap.empty:
    print('Tidak ada kata caption dari daftar heuristik yang muncul di >=5 video grup atas; klaim tidak dibuat.')
else:
    print(kat_cap.head(10).to_string(index=False))
lift_df.to_csv(os.path.join(BASE, 'outputs', '04_lift_interaksi_tinggi.csv'), index=False, encoding='utf-8-sig')
print('Interpretasi: lift > 1 berarti item lebih sering muncul di video ber-like-rate tinggi; ini asosiasi, bukan sebab-akibat.')

In [ ]:
# Sel 8: Pertanyaan 5 - follower vs like rate
bins = [0, 1e4, 1e5, 1e6, np.inf]
labels = ['<10 rb', '10 rb-100 rb', '100 rb-1 jt', '≥1 jt']
df['follower_tier'] = pd.cut(df['author_followers'], bins=bins, labels=labels, right=False)
fol = df.groupby('follower_tier', observed=True).agg(
    n_video=('video_id', 'size'),
    median_views=('views', 'median'),
    median_like_rate=('like_rate', 'median')).reset_index()
fol['median_like_pct'] = (fol['median_like_rate']*100).round(2)
print(fol[['follower_tier','n_video','median_views','median_like_pct']].to_string(index=False))
fol.to_csv(os.path.join(BASE, 'outputs', '04_ringkasan_follower.csv'), index=False, encoding='utf-8-sig')

def spearman(a, b):
    m = a.notna() & b.notna()
    return round(a[m].rank().corr(b[m].rank()), 3)

print('Spearman followers vs like_rate :', spearman(df.author_followers, df.like_rate))
print('Spearman followers vs views     :', spearman(df.author_followers, df.views))

plt.figure()
ax = sns.barplot(data=fol, x='follower_tier', y='median_like_pct', color=PALETTE[1])
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f%%', padding=3)
plt.title('Like Rate Median per Tingkat Follower')
plt.xlabel('Jumlah follower akun')
plt.ylabel('Like rate median (%)')
simpan('04_follower_vs_like_rate.png')
print('Interpretasi: korelasi followers dengan like_rate lemah (rho=0,10) meski kelompok ≥1 jt punya median tertinggi; korelasi bukan sebab-akibat.')

In [ ]:
# Sel 9: Analisis tambahan - 1% video teratas vs sisanya
n_top = math.ceil(len(df) * 0.01)
top = df.nlargest(n_top, 'views')
rest = df.drop(top.index)

def ringkas(g, nama):
    return {'grup': nama, 'n_video': len(g),
            'median_views': g.views.median(), 'median_likes': g.likes.median(),
            'median_like_pct': round(g.like_rate.median()*100, 2),
            'median_durasi': g.duration_sec.median(),
            'median_follower': g.author_followers.median()}

perbandingan = pd.DataFrame([ringkas(top, 'Top 1% views'), ringkas(rest, 'Sisanya')])
print(perbandingan.to_string(index=False))
print('Kontribusi top 1%: ', round(top.views.sum()/df.views.sum()*100, 1), '% total views |',
      round(top.likes.sum()/df.likes.sum()*100, 1), '% total likes')
perbandingan.to_csv(os.path.join(BASE, 'outputs', '04_top1pct.csv'), index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=perbandingan, x='grup', y='median_views', ax=axes[0], color=PALETTE[0])
axes[0].set_title('Median Views')
axes[0].set_xlabel('Grup')
axes[0].set_ylabel('Views (median)')
sns.barplot(data=perbandingan, x='grup', y='median_like_pct', ax=axes[1], color=PALETTE[2])
axes[1].set_title('Median Like Rate')
axes[1].set_xlabel('Grup')
axes[1].set_ylabel('Like rate median (%)')
fig.suptitle('Perbandingan 1% Video Teratas (berdasarkan views) vs Sisanya')
simpan('04_top1pct_vs_sisanya.png')
print('Interpretasi: 8 video teratas menyumbang 29,0% total views; like rate mediannya 5,48% vs 4,84% pada sisanya.')

In [ ]:
# Sel 10: Korelasi Spearman (berbasis peringkat, tahan outlier)
pasangan = [
    ('duration_sec', 'like_rate'), ('views', 'likes'),
    ('author_followers', 'views'), ('author_followers', 'like_rate'),
    ('n_hashtags', 'like_rate'), ('caption_length', 'like_rate'),
]
korelasi = pd.DataFrame([{'pasangan': f'{a} vs {b}',
                          'spearman_rho': spearman(df[a], df[b])} for a, b in pasangan])
print(korelasi.to_string(index=False))
korelasi.to_csv(os.path.join(BASE, 'outputs', '04_korelasi.csv'), index=False, encoding='utf-8-sig')
print('Catatan: korelasi bukan sebab-akibat; like_rate juga bergantung pada usia video saat data diambil.')

## Ringkasan metodologi EDA

- **Median, bukan mean:** mean views 5,6 juta vs median 394.800 pada data awal; beberapa video viral menarik mean naik.
- **Like rate sebagai proxy:** tidak ada `comments`/`shares`, jadi engagement rate definisi awal tidak dihitung.
- **Ambang frekuensi:** hashtag/sound minimal 5 video agar tidak dipengaruhi satu video saja; jam hanya diklaim jika n ≥ 10.
- **Top 1%:** dihitung dari `views` dengan `ceil(1% × 751)` = 8 video.
- **Kategori konten:** heuristik kata kunci buatan sendiri, bukan label resmi dataset.
- **Tidak bisa dijawab:** Q6 (bahasa/wilayah/Indonesia) dan Q7 (jenis interaksi dominan) karena kolomnya tidak ada.